# Exp 02 — Qwen vs. KeyBERT: which one seeds the graph better?

An informal comparison of two competing ways to bootstrap a knowledge graph from a single document. Both routes feed the extracted labels to GLiNER as **entity labels**; only the seed route differs.

| Route | Step 1 — topic discovery | Step 2 — graph extraction |
| --- | --- | --- |
| **A** | KeyBERT — fast, local keyword extraction | GLiNER uses the keywords as entity labels |
| **B** | Qwen3 (via LiteLLM, local Ollama) — LLM concept extraction | GLiNER uses the concepts as entity labels |

## Setup

Test subject: `data/case_1/mortgage.txt` — a first-person narrative of a mortgage servicing dispute (≈1.1 KB, rich in domain entities: lenders, servicers, amounts, legal notices).

## Discussion & takeaways

1. **GLiNER works in both routes.** The graph machinery (entities + relations) runs identically; only the *seed labels* differ. That's the whole experiment.
2. **KeyBERT stays grounded in the document.** Its keywords are surface-level and precise ("summit loan servicing", "disputed late fee"). The graph it feeds is compact and faithful to the text — but narrow.
3. **Qwen reaches a higher abstraction layer.** Its concepts ("Loan Servicing Dispute", "Credit Score Adjustment") are broader, which makes GLiNER tag more and richer spans → a denser graph (21 vs 12 entities).
4. **This is far from a fair fight — nothing is conclusive yet.**
   - Qwen's number of topics was **not constrained** — it was free to emit as many concepts as it wanted, which naturally inflates the entity count.
   - KeyBERT had **no algorithm yet** to decide the *optimal* number of topics — `use_maxsum` returned 5, but that's a heuristic, not a principled choice.
5. **Tentative direction:** Qwen's ability to produce **more abstract terms** looks genuinely promising — abstraction is exactly what you'd want to *lift* a graph beyond raw surface forms. But the two sides need to be compared on equal footing before any conclusion.

## What it changed in the pipeline

The experiment drove two follow-ups:

- **Adaptive KeyBERT** — the "optimal topic count" problem got a principled answer: the elbow of the similarity scores. → [Discovery](../architecture/discovery.md)
- **Deterministic discovery** — a 0.6b model cannot be trusted to ground evidence or relations, so discovery moved to spaCy dependency parsing (LLM-free). The discovery pipeline answers the follow-up question: can a deterministic dependency parse grow the graph from KeyBERT seeds instead? The initial answer, with the mortgage case, is yes — at the cost of surface-level labels ("obtained from") versus the abstract ones an LLM would invent. → [Discovery](../architecture/discovery.md)

## What to try next

- Cap Qwen's concept count (e.g. `max_concepts` in the schema) to make Route B comparable with Route A.
- Run both routes on a *corpus* (not one doc) and compare graph metrics: coverage, density, and precision of relations.
- Evaluate downstream, not upstream: which graph makes retrieval (`kgraph.retriever`) answer real questions better?

# Exp 02: Qwen vs. KeyBERT — which one seeds the graph better? 🧠

A hands-on comparison of **two competing ways to bootstrap a knowledge graph** from a single document:

| Route | Step 1 — topic discovery | Step 2 — graph extraction |
| --- | --- | --- |
| **A** | 🔑 **KeyBERT** — fast, local keyword extraction | 🕸️ **GLiNER** uses the keywords as *entity labels* |
| **B** | 🧠 **Qwen3** (via LiteLLM) — LLM concept extraction | 🕸️ **GLiNER** uses the concepts as *entity labels* |

The whole point is to see how the *choice of topic seed* (document-local keywords vs. abstract LLM concepts) changes what the graph ends up capturing.

> ⚠️ **Exploratory notebook** — the aim is to record observations and build intuition, *not* to crown a winner. The comparison is intentionally informal and nothing here is conclusive yet.

## Setup 🔧

We run inside the `kgraph` virtual environment (uv-managed, `backend/.venv`), so the package is already importable. In this cell we pull together the two extraction stacks plus the tools we need to present the results.

In [1]:
from pathlib import Path

import pandas as pd
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel

from kgraph.llms import LiteLLMClient

## Load the document 📄

The test subject is `mortgage.txt` — a first-person narrative of a **mortgage servicing dispute** (late fees, a foreclosure notice, credit reporting). It's short (≈1.1 KB) but rich in domain entities: lenders, servicers, amounts, and legal notices.

In [2]:
doc_path = Path("../data/case_1/mortgage.txt")
doc = doc_path.read_text()
print(f"{doc_path}: {len(doc)} chars")
print(doc)

../data/case_1/mortgage.txt: 1102 chars
I obtained a mortgage loan of $285,000 from Meridian Home Lending in June 2021 for my property at 142 Oakwood Drive. The loan was originated with a fixed interest rate of 4.25% and a monthly payment of $1,650.
In early 2024, the servicing of my loan was transferred to Summit Loan Servicing without proper notice. Shortly after the transfer, Summit Loan Servicing began charging me a late fee of $75 even though I had made my payment on time.
I disputed the late fee by phone and in writing on March 3rd, but Summit Loan Servicing never responded to my dispute. On March 20th, I received a foreclosure notice from Summit Loan Servicing claiming I had failed to pay two consecutive monthly payments, which is not true.
I contacted Meridian Home Lending directly, and they confirmed the loan was properly paid through their records before the transfer. Despite this, Summit Loan Servicing reported the missed payments to Equifax, damaging my credit score.
I am r

## 1. KeyBERT keywords 🔑

KeyBERT embeds the document and candidate keyphrases, then picks the most representative ones. Here we use **`use_maxsum=True`**, which selects a *diverse* set of keywords (not just the most frequent) — nice for feeding a graph.

The output is **document-local**: phrases that literally appear in the text (*"summit loan servicing"*, *"foreclosure notice"*, *"disputed late fee"*).

In [3]:
embedding_model = SentenceTransformer("../models/all-MiniLM-L6-v2")
kw_model = KeyBERT(model=embedding_model)

keywords = kw_model.extract_keywords(
    doc,
    keyphrase_ngram_range=(1, 3),
    stop_words="english",
    use_maxsum=True)

pd.DataFrame(keywords, columns=["keyword", "score"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,keyword,score
0,disputed late fee,0.4962
1,foreclosure notice,0.5063
2,obtained mortgage loan,0.5138
3,summit loan servicing,0.5258
4,missed payments equifax,0.5436


## 2. Qwen concept extraction 🧠

The same document goes to **Qwen3** (a local Ollama model, called through LiteLLM). Instead of keywords, we ask for *semantic concepts*, and — thanks to `response_format` — the model must answer **exactly** matching the `Concepts` Pydantic schema (strict `additionalProperties: false`).

The output is **abstract**: *"Loan Agreement and Terms"*, *"Credit Reporting"*, *"Foreclosure Notice"* — broader buckets than KeyBERT's surface phrases.

In [4]:
from kgraph.llms.schemas import concepts

client = LiteLLMClient()
concepts = client.chat_structured(
    prompt=f"Infer the main semantic concepts represented in this document:\n\n{doc}",
    model="ollama/qwen3:0.6b",
    schema=concepts.Concepts,
)
for concept in concepts.concepts:
    print(f"- {concept.name}")

- Loan Agreement and Terms,
- Credit Reporting,
- Credit Score,
- Loan Servicing,
- Late Fees,
- Foreclosure Notice,
- Credit Reporting Service,
- Credit Score Adjustment,
- Loan Servicing Dispute,
- Credit Reporting Service Provider,
- Credit Score Adjustment Request


## 3. Route A — KeyBERT topics + GLiNER 🕸️

The KeyBERT keywords become the **entity labels** that GLiNER searches for when building the graph. So the graph is born from the document's own vocabulary.

Observe the outcome: entities like `mortgage loan`, `Meridian Home Lending`, `Summit Loan Servicing`, `late fee` — and relations like `originated_by`, `serviced_by`, `charged`.

In [ ]:
from kgraph.graph import config
from kgraph.ingestion.factory import build_data_source
from kgraph.extractors.gliner import GLiNERGraph

my_config = config.build_pipeline_config("../configs/params.yaml", entities=[key_i for key_i, _ in keywords])
my_config.ner.name = "../models/gliner-relex-large-v0.5"
my_config.data_source.folder= "../data/case_1"
source = build_data_source(my_config.data_source)
documents = source.fetch()
kgraph = GLiNERGraph(my_config)
kgraph.build(documents)

[RawDocument(id='mortgage', content='I obtained a mortgage loan of $285,000 from Meridian Home Lending in June 2021 for my property at 142 Oakwood Drive. The loan was originated with a fixed interest rate of 4.25% and a monthly payment of $1,650.\nIn early 2024, the servicing of my loan was transferred to Summit Loan Servicing without proper notice. Shortly after the transfer, Summit Loan Servicing began charging me a late fee of $75 even though I had made my payment on time.\nI disputed the late fee by phone and in writing on March 3rd, but Summit Loan Servicing never responded to my dispute. On March 20th, I received a foreclosure notice from Summit Loan Servicing claiming I had failed to pay two consecutive monthly payments, which is not true.\nI contacted Meridian Home Lending directly, and they confirmed the loan was properly paid through their records before the transfer. Despite this, Summit Loan Servicing reported the missed payments to Equifax, damaging my credit score.\nI am 

In [16]:
print(f"\nTotal unique entities: {len(kgraph.get_all_entities())}")
for entity in kgraph.get_all_entities():
    print(f"  [{entity.entity_type}] {entity.text} (mentions: {len(entity.mentions)})")

print(f"\nTotal relations: {len(kgraph.relations)}")
for rel in kgraph.relations:
    print(f"  {rel.head_text} --[{rel.relation_type}]--> {rel.tail_text}")


Total unique entities: 12
  [obtained mortgage loan] mortgage loan (mentions: 1)
  [obtained mortgage loan] $285,000 (mentions: 1)
  [summit loan servicing] Meridian Home Lending (mentions: 1)
  [summit loan servicing] Summit Loan Servicing (mentions: 1)
  [disputed late fee] late fee (mentions: 1)
  [disputed late fee] late fee of $75 (mentions: 1)
  [disputed late fee] $75 (mentions: 1)
  [foreclosure notice] foreclosure notice (mentions: 1)
  [missed payments equifax] two consecutive monthly payments (mentions: 1)
  [missed payments equifax] missed payments (mentions: 1)
  [missed payments equifax] Equifax (mentions: 1)
  [disputed late fee] late fees (mentions: 1)

Total relations: 13
  mortgage loan --[originated_by]--> Meridian Home Lending
  mortgage loan --[secured_by]--> Meridian Home Lending
  mortgage loan --[serviced_by]--> Summit Loan Servicing
  $285,000 --[originated_by]--> Meridian Home Lending
  $285,000 --[secured_by]--> Meridian Home Lending
  $285,000 --[serviced_b

## 4. Route B — Qwen topics + GLiNER 🕸️

Now the same trick, but seeded with **Qwen's abstract concepts** instead of keywords. GLiNER ends up tagging more granular spans (e.g. `damaging my credit score`, `requesting that the late fees be reversed`) — because the concepts are more general, more text falls under them.

Compare the two graphs side by side: Route A found **12 entities / 13 relations**, Route B **21 entities / 15 relations**.

In [20]:
my_config = config.build_pipeline_config("../configs/params.yaml", entities=[concept.name for concept in concepts.concepts])
my_config.ner.name = "../models/gliner-relex-large-v0.5"
my_config.data_source.folder= "../data/case_1"
kgraph = GLiNERGraph(my_config)
kgraph.build(documents)

print(f"\nTotal unique entities: {len(kgraph.get_all_entities())}")
for entity in kgraph.get_all_entities():
    print(f"  [{entity.entity_type}] {entity.text} (mentions: {len(entity.mentions)})")

print(f"\nTotal relations: {len(kgraph.relations)}")
for rel in kgraph.relations:
    print(f"  {rel.head_text} --[{rel.relation_type}]--> {rel.tail_text}")

Extracted 21 entities and 15 relations from mortgage

Total unique entities: 21
  [Loan Agreement and Terms,] mortgage loan (mentions: 1)
  [Credit Score,] $285,000 (mentions: 1)
  [Credit Reporting Service Provider,] Meridian Home Lending (mentions: 1)
  [Loan Agreement and Terms,] 4.25% (mentions: 1)
  [Loan Agreement and Terms,] $1,650 (mentions: 1)
  [Loan Servicing,] Summit Loan Servicing (mentions: 1)
  [Late Fees,] late fee (mentions: 1)
  [Late Fees,] late fee of $75 (mentions: 1)
  [Late Fees,] $75 (mentions: 1)
  [Loan Servicing Dispute,] disputed (mentions: 1)
  [Loan Servicing Dispute,] disputed the late fee (mentions: 1)
  [Loan Servicing Dispute,] dispute (mentions: 1)
  [Foreclosure Notice,] foreclosure notice (mentions: 1)
  [Credit Reporting Service,] Equifax (mentions: 1)
  [Credit Score Adjustment,] damaging (mentions: 1)
  [Credit Score Adjustment,] damaging my credit score (mentions: 1)
  [Credit Score,] my credit score (mentions: 1)
  [Credit Score,] credit score 

## Discussion & takeaways ✍️

Looking at both routes side by side, a few things stand out — but let's be honest about what they mean:

1. **🕸️ GLiNER works in both routes.** The graph machinery (entities + relations) runs identically; only the *seed labels* differ. That's the whole experiment.
2. **🔑 KeyBERT stays grounded in the document.** Its keywords are surface-level and precise (*"summit loan servicing"*, *"disputed late fee"*). The graph it feeds is compact and faithful to the text — but narrow.
3. **🧠 Qwen reaches a higher abstraction layer.** Its concepts (*"Loan Servicing Dispute"*, *"Credit Score Adjustment"*) are broader, which makes GLiNER tag more and richer spans → a denser graph (21 vs 12 entities).
4. **⚠️ But this is far from a fair fight — nothing is conclusive yet.**
   - Qwen's number of topics was **not constrained** — it was free to emit as many concepts as it wanted, which naturally inflates the entity count.
   - KeyBERT, on the other hand, has **no algorithm yet** to decide the *optimal* number of topics — `use_maxsum` returned 5, but that's a heuristic, not a principled choice.
5. **💡 Tentative direction:** the fact that Qwen is capable of producing **more abstract terms** looks genuinely promising — abstraction is exactly what you'd want to *lift* a graph beyond raw surface forms. But we need to compare on equal footing before drawing any conclusion.

### What to try next 🎯

- Cap Qwen's concept count (e.g. `max_concepts` in the schema) to make Route B comparable with Route A.
- Add an **optimal-topic-count heuristic** for KeyBERT (elbow method on MMR scores, silhouette over embeddings, etc.).
- Run both routes on a *corpus* (not one doc) and compare graph metrics: coverage, density, and precision of relations.
- Evaluate downstream, not upstream: which graph makes retrieval (`kgraph.retriever`) answer real questions better?

**Bottom line:** the experiment setup works end-to-end and the signals are intriguing — especially Qwen's abstraction. But until both sides are tuned symmetrically, the verdict stays open. 🧪